# Day 2 workspace export (`d2_rag_eval`)
Saves everything this Day 2 workspace needs to be picked up again, in this account or your own, with no AI calls.

| Output | Where | Goes into git? |
| --- | --- | --- |
| Day 2 tables and views as CSV (VARIANT / ARRAY / OBJECT / VECTOR as JSON text), DDL, stage manifest, prerequisites, `SUMMARY.md` | `results/<YYYY-MM-DD>/` in this workspace | yes |
| Copies of the Day 2 stage files (slide PDFs, slide images, lesson videos) | `@LAB_EXPORT_STAGE_D2_RAG_EVAL/<date>/stage_files/` | no |
| Media zip of those files (download and keep next to the repo) | `@LAB_EXPORT_STAGE_D2_RAG_EVAL/<date>/lab_export_d2_rag_eval_media_<date>.zip` | no (course media, about 2.7 GB) |
| Code zip of the workspace as git sees it (`.gitignore` applies), including `results/` | `@LAB_EXPORT_STAGE_D2_RAG_EVAL/<date>/lab_export_d2_rag_eval_<date>.zip` | it is the repo |

**Day 2 only.** An object is exported when its name appears in this workspace's code (`sql/`, `python/`, `lab_metadata/runbook_lib.py`) or it is a runbook experiment snapshot (`<table>_<run label>`). Everything else in the schema (Day 1) is listed as excluded and never exported.

Run the cells top to bottom. To restore, use `workspace_restore.ipynb`.

In [ ]:
import datetime

PROJECT = "d2_rag_eval"                                   # Day 2 RAG evaluation pipeline
EXPORT_DATE = datetime.date.today().isoformat()            # results/<date>/ and the stage folder <date>/
EXPORT_STAGE = f"LAB_EXPORT_STAGE_{PROJECT.upper()}"       # created here; an existing stage of another owner/purpose is never reused
STAGE_COMMENT = f"{PROJECT} workspace export (created by lab_metadata/workspace_export.ipynb)"
MAKE_MEDIA_ZIP = True        # also build the downloadable media zip (about 2.7 GB); the stage copies are made either way
OVERWRITE_DATE = False       # True = replace an export already made today (results/<date>/ and the stage folder)

In [ ]:
import importlib, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from snowflake.snowpark.context import get_active_session

WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql" / "runbook_config.sql").exists())
sys.path.insert(0, str(WORKSPACE / "lab_metadata"))
import runbook_lib, export_lib
importlib.reload(runbook_lib); importlib.reload(export_lib)
pd.set_option("display.max_colwidth", 300)

session = get_active_session()
rb = runbook_lib.Runbook(session, WORKSPACE, {})
rb.apply_config()                      # same session context and values as the pipeline (sql/runbook_config.sql)
DB, SCHEMA = rb.config["database"].upper(), rb.config["schema"].upper()

included, excluded = export_lib.day2_scope(session, WORKSPACE, DB, SCHEMA, exclude_stages=[EXPORT_STAGE])
display(Markdown(f"**Source** `{DB}.{SCHEMA}` · workspace `{WORKSPACE.name}` · export date `{EXPORT_DATE}`"))
display(export_lib.scope_frame(included, excluded))

## Save results (run first)
Writes `results/<date>/`: one CSV per Day 2 table and view, the DDL of every Day 2 object, the stage manifest (path, size, MD5 of every file), `prerequisites.md`, and `SUMMARY.md` (scope, latest batch, champion, check pass counts, data-quality checks, row count of every file, how to restore). No AI calls.

In [ ]:
import json, shutil
importlib.reload(export_lib)

OUT = WORKSPACE / "results" / EXPORT_DATE
if OUT.exists() and not OVERWRITE_DATE:
    raise RuntimeError(f"{OUT} already exists. Set OVERWRITE_DATE = True to replace today's export.")
shutil.rmtree(OUT, ignore_errors=True)
for sub in ("tables", "views", "ddl"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
fq = lambda name: f"{DB}.{SCHEMA}.{name}"
files, failures, types = [], [], {}

# 1. Tables and views: CSV (split into NAME.partNN.csv when over 20 MB), exact column types, DDL
for kind in ("table", "view"):
    ddl = []
    for name in included.get(kind, []):
        try:
            n, cols, fnames = export_lib.export_relation(session, fq(name), kind, OUT / f"{kind}s")
            types[name] = {"kind": kind, "columns": cols, "files": fnames}
            ddl.append(export_lib.table_ddl(session, fq(name), kind.upper()).rstrip().rstrip(";") + ";")
            files.append({"file": " + ".join(f"{kind}s/{f}" for f in fnames), "rows": n})
        except Exception as exc:
            failures.append(f"{kind} {name}: {str(exc)[:200]}")
    (OUT / "ddl" / f"{kind}s.sql").write_text("\n\n".join(ddl) + "\n")
(OUT / "ddl" / "column_types.json").write_text(json.dumps(types, indent=1))

# 2. Functions, procedures, stages, Cortex Search services
for kind in ("function", "procedure"):
    text = export_lib.routine_ddls(session, DB, SCHEMA, kind, included.get(kind, []))
    (OUT / "ddl" / f"{kind}s.sql").write_text("\n\n".join(t.rstrip().rstrip(";") + ";" for t in text) + "\n")
(OUT / "ddl" / "stages.sql").write_text("\n\n".join(export_lib.stage_ddl(session, DB, SCHEMA, s) for s in included.get("stage", [])) + "\n")
services = [export_lib.search_service_ddl(session, DB, SCHEMA, s) for s in included.get("cortex_search_service", [])]
(OUT / "ddl" / "cortex_search_services.sql").write_text("\n\n".join(s[0] for s in services) + "\n")
for name in ("tables", "views", "functions", "procedures", "stages", "cortex_search_services", "column_types"):
    files.append({"file": f"ddl/{name}.{'json' if name == 'column_types' else 'sql'}", "rows": None})

# 3. Stage manifest (the media itself is copied to the export stage in the next cell)
manifest = export_lib.stage_manifest(session, DB, SCHEMA, included.get("stage", []))
manifest.to_csv(OUT / "stage_manifest.csv", index=False)
files.append({"file": "stage_manifest.csv", "rows": len(manifest)})

# 4. Scope and prerequisites
export_lib.scope_frame(included, excluded).to_csv(OUT / "scope.csv", index=False)
files.append({"file": "scope.csv", "rows": None})
prereq = ["# Prerequisites for a restore (values are never exported)", "",
          "| Object | What to create |", "| --- | --- |"]
prereq += [f"| {k} | {v} |" for k, v in export_lib.PREREQUISITES.items()]
prereq += [f"| secret in source schema | {s} (recreate with your own key: `CREATE SECRET dataexpert_api_key TYPE = GENERIC_STRING SECRET_STRING = '<key>';`) |" for s in included.get("secret", [])]
prereq += [f"| warehouse | {rb.config['warehouse']} (any warehouse; set it in sql/runbook_config.sql) |",
           f"| role | {rb.config['role']} (needs CREATE SCHEMA/TABLE/VIEW/STAGE/PROCEDURE and CORTEX SEARCH SERVICE, and SNOWFLAKE.CORTEX_USER) |",
           "| models | " + ", ".join(sorted({rb.config.get(k) for k in ('text_embed_model', 'image_embed_model', 'describe_model', 'answer_model', 'judge_model')} - {None})) + " (may need cross-region inference) |",
           "", "Without the secret and the external access integration everything restores and the pipeline runs on the exported data, ",
           "but ingest (steps 02, 03, 12, 13b) cannot fetch new lessons: run the runbook with `REFRESH_SOURCES=False`."]
(OUT / "prerequisites.md").write_text("\n".join(prereq) + "\n")
files.append({"file": "prerequisites.md", "rows": None})

# 5. SUMMARY.md
latest, champ, passes, verdicts = export_lib.run_overview(session)
dq = export_lib.data_quality(session, rb)
cell = lambda v: "" if v is None or (isinstance(v, float) and v != v) else str(int(v)) if isinstance(v, float) and v.is_integer() else str(v)
md = lambda df: "\n".join(["| " + " | ".join(map(str, df.columns)) + " |", "| " + " | ".join("---" for _ in df.columns) + " |"]
                          + ["| " + " | ".join(cell(v) for v in r) + " |" for r in df.astype(object).itertuples(index=False)])
stage_sizes = manifest.groupby("stage").agg(files=("relative_path", "count"), mb=("size", lambda s: round(s.sum() / 1e6, 1))).reset_index()
summary = f"""# Day 2 export summary: {PROJECT}, {EXPORT_DATE}

## Scope
- Source: `{DB}.{SCHEMA}`, this Day 2 workspace. Day 2 only; Day 1 objects are excluded (see `scope.csv`).
- Exported: {sum(len(v) for v in included.values())} objects ({', '.join(f'{len(v)} {k}' for k, v in included.items())}).
- Excluded: {sum(len(v) for v in excluded.values())} objects: {'; '.join(f'{k}: ' + ', '.join(v) for k, v in excluded.items())}.
- CSVs over 20 MB are split into plain-text parts (`NAME.part01.csv`, ...). VARIANT, ARRAY, OBJECT and VECTOR columns are JSON text; `\\N` marks a SQL NULL.
- Floats are written with 16 significant digits (the most Snowflake text output carries); everything else round-trips exactly.

## Latest batch
- Run `{latest.get('RUN_ID')}`: {latest.get('STATUS')}, mode {latest.get('MODE')}, experiment `{latest.get('EXPERIMENT_NAME')}`, started {latest.get('STARTED_AT')}.
- Change note: {latest.get('CHANGE_NOTE')}
- Verdicts:
{chr(10).join('  - ' + v for v in verdicts) or '  - none'}

## Champion
- Run `{champ.get('RUN_ID')}` (`{champ.get('EXPERIMENT_NAME')}`), promoted {champ.get('PROMOTED_AT')}: {champ.get('PROMOTED_REASON')}

## Check pass counts
{md(passes) if len(passes) else 'no checks recorded'}

## Data-quality checks (expect 0 bad rows)
{md(dq)}

## Stage files (copied to @{EXPORT_STAGE}/{EXPORT_DATE}/stage_files/)
{md(stage_sizes)}

## Files in results/{EXPORT_DATE}/
{md(pd.DataFrame(files))}

## Failures
{chr(10).join('- ' + f for f in failures) or 'none'}

## How to restore
1. Create the prerequisites in `prerequisites.md` (secret with your key, network rule, external access integration).
2. Upload the media zip's folders to a stage (for example with `snow stage copy`), or use `@{EXPORT_STAGE}/{EXPORT_DATE}/stage_files/` in this account.
3. Open `lab_metadata/workspace_restore.ipynb`, set your role, database, schema, warehouse, `RESULTS_DATE = "{EXPORT_DATE}"` and the media location, and run it top to bottom.
4. It creates the tables, loads these CSVs, copies the media into the stages, rebuilds views, functions, procedures and the search service, and records the restored state as the champion.
5. Run `pipeline_runbook.ipynb`: every step should skip.
"""
(OUT / "SUMMARY.md").write_text(summary)
written = [p for p in OUT.rglob("*") if p.is_file()]
missing = [f for t in types.values() for f in t["files"] if not (OUT / f"{t['kind']}s" / f).exists()]
if missing:
    raise RuntimeError(f"{len(missing)} CSV files were not written: {missing[:5]}")
display(Markdown(f"Wrote `results/{EXPORT_DATE}/`: {len(written)} files, {sum(f['rows'] or 0 for f in files)} CSV rows, "
                 f"largest file {max(p.stat().st_size for p in written) / 1e6:.1f} MB; failures: {len(failures)}"))
display(dq)
if failures:
    display(Markdown("**Failures:**\n" + "\n".join(f"- {f}" for f in failures)))

## Copy the Day 2 stage files into the export stage
Creates `@LAB_EXPORT_STAGE_D2_RAG_EVAL` (only if it does not exist; a stage with that name that this notebook did not create is never reused) and copies every Day 2 stage into `<date>/stage_files/<stage>/` with `COPY FILES`, inside Snowflake. Then every file's size and MD5 is checked against the manifest.

In [ ]:
existing = [r.as_dict() for r in session.sql(f"SHOW STAGES LIKE '{EXPORT_STAGE}' IN SCHEMA {DB}.{SCHEMA}").collect()]
if existing and (existing[0].get("comment") or "") != STAGE_COMMENT:
    raise RuntimeError(f"{EXPORT_STAGE} already exists and was not created by this notebook (comment: {existing[0].get('comment')!r}). "
                       "Choose another PROJECT name; an existing stage is never reused.")
if not existing:
    session.sql(f"""CREATE STAGE {DB}.{SCHEMA}.{EXPORT_STAGE} DIRECTORY = (ENABLE = TRUE)
                    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') COMMENT = '{STAGE_COMMENT}'""").collect()
ES = f"@{DB}.{SCHEMA}.{EXPORT_STAGE}"
prior = session.sql(f"LIST {ES}/{EXPORT_DATE}/").collect()
if prior and not OVERWRITE_DATE:
    raise RuntimeError(f"{ES}/{EXPORT_DATE}/ already has {len(prior)} files. Set OVERWRITE_DATE = True to replace them.")
if prior:
    session.sql(f"REMOVE {ES}/{EXPORT_DATE}/").collect()

for stage in included.get("stage", []):
    session.sql(f"COPY FILES INTO {ES}/{EXPORT_DATE}/stage_files/{stage}/ FROM @{DB}.{SCHEMA}.{stage}/").collect()
session.sql(f"ALTER STAGE {DB}.{SCHEMA}.{EXPORT_STAGE} REFRESH").collect()

copied = pd.DataFrame([r.as_dict() for r in session.sql(
    f"SELECT relative_path, size, md5 FROM DIRECTORY({ES}) WHERE STARTSWITH(relative_path, ?)",
    params=[f"{EXPORT_DATE}/stage_files/"]).collect()])
copied["stage"] = copied.RELATIVE_PATH.str.split("/").str[2]
copied["relative_path"] = copied.RELATIVE_PATH.str.split("/", n=3).str[3]
check = manifest.merge(copied[["stage", "relative_path", "SIZE", "MD5"]], on=["stage", "relative_path"], how="left")
check["ok"] = (check["size"] == check["SIZE"]) & (check["md5"] == check["MD5"])
by_stage = check.groupby("stage").agg(files=("ok", "size"), matched=("ok", "sum"), mb=("size", lambda s: round(s.sum() / 1e6, 1))).reset_index()
display(by_stage)
if not check.ok.all():
    raise RuntimeError(f"{(~check.ok).sum()} files did not copy exactly: " + ", ".join(check[~check.ok].relative_path.head(10)))
display(Markdown(f"All {len(check)} Day 2 stage files are in `{ES}/{EXPORT_DATE}/stage_files/` with matching size and MD5."))

## Media zip (download and keep next to the repo)
Built one file at a time from the copies above (download, add, delete), so the notebook's disk only holds the zip plus one video. Videos and images are stored uncompressed (they do not compress). Course media stays out of git (`.gitignore` excludes `*.pdf` and `*.png`).

In [ ]:
MEDIA_ZIP = Path(f"/tmp/lab_export_{PROJECT}_media_{EXPORT_DATE}.zip")
if MAKE_MEDIA_ZIP:
    need = int(manifest["size"].sum() * 1.05 + manifest["size"].max())
    free = shutil.disk_usage("/tmp").free
    if free < need:
        raise RuntimeError(f"Not enough disk in /tmp for the media zip: {free / 1e9:.1f} GB free, {need / 1e9:.1f} GB needed. "
                           f"The media is still safe in {ES}/{EXPORT_DATE}/stage_files/; set MAKE_MEDIA_ZIP = False to skip.")
    MEDIA_ZIP.unlink(missing_ok=True)
    n, size, bad = export_lib.zip_stage_files(session, manifest, f"{ES}/{EXPORT_DATE}/stage_files", MEDIA_ZIP, Path("/tmp/d2_media"))
    if bad:
        raise RuntimeError(f"{len(bad)} files in the zip do not match the manifest size: {bad[:5]}")
    session.file.put(str(MEDIA_ZIP), f"{ES}/{EXPORT_DATE}/", auto_compress=False, overwrite=True)
    MEDIA_ZIP.unlink()
    display(Markdown(f"Media zip: {n} files, {size / 1e6:,.0f} MB, uploaded to `{ES}/{EXPORT_DATE}/{MEDIA_ZIP.name}`"))
else:
    display(Markdown("MAKE_MEDIA_ZIP = False: media zip skipped (the stage copies above are the media export)."))

## Code zip (the workspace as git sees it, including `results/`)

In [ ]:
CODE_ZIP = Path(f"/tmp/lab_export_{PROJECT}_{EXPORT_DATE}.zip")
CODE_ZIP.unlink(missing_ok=True)
n, size = export_lib.zip_workspace(WORKSPACE, CODE_ZIP)
session.file.put(str(CODE_ZIP), f"{ES}/{EXPORT_DATE}/", auto_compress=False, overwrite=True)
CODE_ZIP.unlink()
display(Markdown(f"Code zip: {n} files, {size / 1e6:.1f} MB, uploaded to `{ES}/{EXPORT_DATE}/{CODE_ZIP.name}`"))

In [ ]:
listing = pd.DataFrame([r.as_dict() for r in session.sql(f"LIST {ES}/{EXPORT_DATE}/").collect()])
listing["folder"] = listing["name"].str.split("/").apply(lambda p: "/".join(p[2:4]) if len(p) > 4 else p[-1])
display(listing.groupby("folder").agg(files=("name", "count"), mb=("size", lambda s: round(s.sum() / 1e6, 1))).reset_index())
display(Markdown(f"""**Download** from Snowsight: Data » Databases » {DB} » {SCHEMA} » Stages » {EXPORT_STAGE} » `{EXPORT_DATE}/`
- `lab_export_{PROJECT}_{EXPORT_DATE}.zip`: the code and `results/{EXPORT_DATE}/` (also commit `results/{EXPORT_DATE}/` to git)
- `lab_export_{PROJECT}_media_{EXPORT_DATE}.zip`: slide PDFs, slide images and videos (keep next to the repo, not in git).
  Large files may be easier to fetch with the Snowflake CLI: `snow stage copy {ES}/{EXPORT_DATE}/ ./export/`"""))